# Real-time Object Tracking (YOLO + ByteTrack/BoT-SORT)

Notebook-first refactor for multi-object tracking on short videos with real dataset download, validation, tracking inference, and honest evaluation outputs.

## Dataset Source

Kaggle dataset: https://www.kaggle.com/datasets/mistag/short-videos

This notebook downloads assets from Kaggle in code, validates video files, checks for available labels/splits, and then runs real tracking inference.

In [ ]:
import importlib
import subprocess
import sys

def ensure_package(import_name, pip_name=None):
    package_name = pip_name or import_name
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package_name])

ensure_package('kagglehub')
ensure_package('cv2', 'opencv-python-headless')
ensure_package('numpy')
ensure_package('pandas')
ensure_package('matplotlib')
ensure_package('torch')
ensure_package('ultralytics')
print('Dependencies are ready.')

In [ ]:
import json
import os
import random
import time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
PROJECT_DIR = Path.cwd()
ARTIFACTS_DIR = PROJECT_DIR / 'artifacts'
RUNS_DIR = ARTIFACTS_DIR / 'tracking_runs'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Device: {DEVICE}')
print(f'Project directory: {PROJECT_DIR}')

## Real Dataset Download From Kaggle

In [ ]:
import kagglehub

if not os.getenv('KAGGLE_USERNAME') or not os.getenv('KAGGLE_KEY'):
    raise EnvironmentError('Missing Kaggle credentials. Set KAGGLE_USERNAME and KAGGLE_KEY.')

dataset_root = Path(kagglehub.dataset_download('mistag/short-videos'))
print(f'Dataset root: {dataset_root}')

## Dataset Verification: Files, Labels, and Splits

In [ ]:
video_exts = {'.mp4', '.avi', '.mov', '.mkv', '.webm'}
video_paths = [p for p in dataset_root.rglob('*') if p.suffix.lower() in video_exts]
if len(video_paths) == 0:
    raise RuntimeError('No video files found in downloaded dataset.')

verified_rows = []
for vp in video_paths:
    cap = cv2.VideoCapture(str(vp))
    ok = cap.isOpened()
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) if ok else 0
    fps = float(cap.get(cv2.CAP_PROP_FPS)) if ok else 0.0
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) if ok else 0
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) if ok else 0
    cap.release()
    if ok and frames > 0 and w > 0 and h > 0:
        verified_rows.append({
            'video_path': str(vp),
            'frames': frames,
            'fps': fps,
            'width': w,
            'height': h
        })

video_df = pd.DataFrame(verified_rows)
if len(video_df) == 0:
    raise RuntimeError('Video integrity verification failed for all files.')

label_files = []
for ext in ['*.txt', '*.json', '*.xml', '*.csv']:
    label_files.extend(list(dataset_root.rglob(ext)))

split_counts = {'train': 0, 'val': 0, 'test': 0, 'unknown': 0}
def infer_split(path_str):
    p = path_str.lower()
    if 'train' in p:
        return 'train'
    if 'val' in p or 'valid' in p or 'dev' in p:
        return 'val'
    if 'test' in p:
        return 'test'
    return 'unknown'

video_df['split'] = video_df['video_path'].apply(infer_split)
for s in video_df['split'].tolist():
    split_counts[s] += 1

if split_counts['train'] == 0 and split_counts['val'] == 0 and split_counts['test'] == 0:
    shuffled = video_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    n = len(shuffled)
    n_train = int(0.7 * n)
    n_val = int(0.15 * n)
    shuffled.loc[:n_train - 1, 'split'] = 'train'
    shuffled.loc[n_train:n_train + n_val - 1, 'split'] = 'val'
    shuffled.loc[n_train + n_val:, 'split'] = 'test'
    video_df = shuffled
    split_counts = video_df['split'].value_counts().to_dict()

print(f'Verified videos: {len(video_df)}')
print(f'Label/annotation files found: {len(label_files)}')
print(f'Split counts: {split_counts}')

if split_counts.get('train', 0) == 0 or split_counts.get('val', 0) == 0 or split_counts.get('test', 0) == 0:
    raise RuntimeError('Split verification failed: train/val/test all required.')

display(video_df.head(10))

## Tracking Model Setup (YOLO + ByteTrack/BoT-SORT)

In [ ]:
from ultralytics import YOLO

preferred_weights = ['yolo26s.pt', 'yolo26m.pt', 'yolo11n.pt', 'yolov8n.pt']
selected_weights = None
for w in preferred_weights:
    try:
        model = YOLO(w)
        selected_weights = w
        break
    except Exception:
        continue

if selected_weights is None:
    raise RuntimeError('Could not load any YOLO weights for tracking.')

print(f'Using weights: {selected_weights}')

In [ ]:
val_videos = video_df[video_df['split'] == 'val'].copy().reset_index(drop=True)
if len(val_videos) == 0:
    raise RuntimeError('No validation videos available after split verification.')

eval_video = val_videos.iloc[0]['video_path']
print(f'Evaluation video: {eval_video}')

## Real Tracking Inference and Quantitative Runtime Metrics

In [ ]:
def run_tracker(video_path, tracker_name, run_name):
    start = time.time()
    frame_count = 0
    det_count_total = 0
    tracked_ids = set()

    results_stream = model.track(
        source=video_path,
        tracker=tracker_name,
        persist=True,
        stream=True,
        save=True,
        project=str(RUNS_DIR),
        name=run_name,
        device=0 if DEVICE == 'cuda' else 'cpu',
        verbose=False
    )

    for res in results_stream:
        frame_count += 1
        if res.boxes is None:
            continue
        n = len(res.boxes)
        det_count_total += int(n)
        if hasattr(res.boxes, 'id') and res.boxes.id is not None:
            ids = res.boxes.id.cpu().numpy().astype(int).tolist()
            for tid in ids:
                tracked_ids.add(int(tid))

    elapsed = max(1e-6, time.time() - start)
    out_video = RUNS_DIR / run_name / Path(video_path).name

    return {
        'tracker': tracker_name,
        'run_name': run_name,
        'video_path': video_path,
        'output_video': str(out_video),
        'frames_processed': int(frame_count),
        'detections_total': int(det_count_total),
        'unique_track_ids': int(len(tracked_ids)),
        'elapsed_seconds': float(elapsed),
        'fps_runtime': float(frame_count / elapsed),
        'mean_detections_per_frame': float(det_count_total / frame_count) if frame_count > 0 else 0.0
    }

In [ ]:
metric_rows = []
metric_rows.append(run_tracker(eval_video, 'bytetrack.yaml', 'bytetrack_run'))
metric_rows.append(run_tracker(eval_video, 'botsort.yaml', 'botsort_run'))
metrics_df = pd.DataFrame(metric_rows)
display(metrics_df)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].bar(metrics_df['tracker'], metrics_df['fps_runtime'])
axes[0].set_title('Runtime FPS')
axes[0].set_ylabel('fps')

axes[1].bar(metrics_df['tracker'], metrics_df['unique_track_ids'])
axes[1].set_title('Unique Track IDs')

axes[2].bar(metrics_df['tracker'], metrics_df['mean_detections_per_frame'])
axes[2].set_title('Mean Detections / Frame')

plt.tight_layout()
comparison_png = ARTIFACTS_DIR / 'tracker_comparison.png'
plt.savefig(comparison_png, dpi=140)
plt.show()

## Honest Qualitative Review

In [ ]:
preview_rows = []
for out_path in metrics_df['output_video'].tolist():
    cap = cv2.VideoCapture(out_path)
    ok, frame = cap.read()
    cap.release()
    if ok:
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        preview_rows.append((out_path, frame_rgb))

fig, axes = plt.subplots(1, len(preview_rows), figsize=(7 * max(1, len(preview_rows)), 5))
if len(preview_rows) == 1:
    axes = [axes]

for ax, (name, frame_rgb) in zip(axes, preview_rows):
    ax.imshow(frame_rgb)
    ax.set_title(Path(name).parent.name)
    ax.axis('off')

plt.tight_layout()
qualitative_png = ARTIFACTS_DIR / 'qualitative_first_frames.png'
plt.savefig(qualitative_png, dpi=140)
plt.show()

## Save Real Outputs

In [ ]:
video_stats_csv = ARTIFACTS_DIR / 'dataset_video_stats.csv'
video_df.to_csv(video_stats_csv, index=False)

tracking_metrics_csv = ARTIFACTS_DIR / 'tracking_metrics.csv'
metrics_df.to_csv(tracking_metrics_csv, index=False)

summary_metrics = {
    'dataset': 'mistag/short-videos',
    'video_count_verified': int(len(video_df)),
    'label_file_count': int(len(label_files)),
    'split_counts': {k: int(v) for k, v in video_df['split'].value_counts().to_dict().items()},
    'weights_used': selected_weights,
    'trackers_evaluated': metrics_df['tracker'].tolist(),
    'best_runtime_fps': float(metrics_df['fps_runtime'].max()),
    'max_unique_track_ids': int(metrics_df['unique_track_ids'].max())
}

metrics_json = ARTIFACTS_DIR / 'metrics.json'
with open(metrics_json, 'w', encoding='utf-8') as f:
    json.dump(summary_metrics, f, indent=2)

manifest = {
    'dataset_url': 'https://www.kaggle.com/datasets/mistag/short-videos',
    'dataset_root': str(dataset_root),
    'video_stats_csv': str(video_stats_csv),
    'tracking_metrics_csv': str(tracking_metrics_csv),
    'metrics_json': str(metrics_json),
    'comparison_plot_png': str(comparison_png),
    'qualitative_png': str(qualitative_png),
    'tracker_outputs': metrics_df['output_video'].tolist()
}
manifest_json = ARTIFACTS_DIR / 'artifact_manifest.json'
with open(manifest_json, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2)

print('Saved outputs:')
print(f'- {video_stats_csv}')
print(f'- {tracking_metrics_csv}')
print(f'- {metrics_json}')
print(f'- {manifest_json}')
print(f'- {comparison_png}')
print(f'- {qualitative_png}')

## Limitations and Next Steps

- This dataset may not provide MOT ground-truth IDs in a standard benchmark format, so this notebook reports runtime/track statistics and qualitative analysis instead of MOTA/HOTA.
- For full benchmark metrics, use a tracking dataset with official identity annotations and evaluate with a MOT metrics toolkit.
- To improve tracking quality, tune confidence/IoU thresholds and compare larger model weights on the same validation videos.